## Intelligent Research & Audit Agent

**Objective:** Build a tool that takes a complex topic (e.g., "The impact of 2024 AI regulations on healthcare startups"), researches it across multiple sources (web + provided docs), and produces a structured report.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

**Phase 1:** Agentic loop creation with langgraph
**Nodes**:
- Planner: Breaks question down into multiple sub-questions
- Search: Searches internet using `tavily` for each sub-questions
- Writer: From search results creates a final report 
- Reviewer: Reviews the final report and provides feedback. Loops back to Writer node until report is considered good.

**Node 1: Planner**

In [ ]:
# Define state. State is a shared dictionary/memory which all node read/write to.
from typing import List, TypedDict

class ResearchState(TypedDict):
	question: str
	plan: List[str]
	report_sections: List[str]
	final_report: str
	feedback: str
	loop_count: int

In [ ]:
# Define pydantic model to get `Structured Output` from llm
from pydantic import BaseModel, Field

class Plan(BaseModel):
    """Series of research steps to answer a user's complex question"""
    steps: List[str] = Field(description="Individual research tasks or question to investigate")

In [ ]:
# Define planner_node
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.types import interrupt


# initialize llm
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
planner_llm = llm.with_structured_output(Plan)

def planner_node(state: ResearchState):
	"""Analyze the question and create a research plan"""
	if not state.get("plan"):
		prompt = f"""You are a research expert. 
		Break down the following complex query into 3-4 specific research tasks.
		Query: {state['question']}"""

		# llm response will be of Plan object
		response = planner_llm.invoke(prompt)
		initial_steps = response.steps
	else:
		initial_steps = state["plan"]

	user_input = interrupt({
        "message": "Please review the research plan:",
        "plan": initial_steps
    })
	if isinstance(user_input, list):
		final_steps = user_input
	elif isinstance(user_input, dict) and "updated_plan" in user_input:
		final_steps = user_input["updated_plan"]
	else:
		final_steps = initial_steps
	return {"plan": final_steps}

**Node 2: Search**

In [ ]:
# Setup tavily for llm friendly search results
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results = 5)

In [ ]:
# Define search_node
def search_node(state: ResearchState):
    """Execute search for each step in the plan and stores in results"""
    all_results = []
    print(f"--- Executing Search for {len(state['plan'])} Steps ---")
    for step in state['plan']:
        print(f"Searching for: {step}")
        results = search_tool.invoke({"query": step})
        results = results['results'] 
        combined_content = "\n".join([r["content"] for r in results])
        all_results.append({'step': step, 'content': combined_content})
    return {"report_sections": all_results}

**Node 3: writer_node**

In [ ]:
# Define writer_node
def writer_node(state: ResearchState):
    """Synthesize search results into a final report"""
    print("--- Generating final report ---")
    context = ""
    for entry in state['report_sections']:
        context += f"\n\nSource Topic: {entry['step']}\nContent: {entry['content']}"
    prompt = f"""You are a professional research analyst. 
    Based on the following research data, write a comprehensive report answering the original question.
    
    Original Question: {state['question']}
    
    Research Data:
    {context}
    "Earlier feedback for improvement: {state.get('feedback', 'None')}"
    Instructions:
    - Use Markdown formatting (headers, bullet points).
    - Be factual and concise.
    - If information is missing, acknowledge it.
    """
    response = llm.invoke(prompt)
    return {"final_report":  response.content}

**Node 4: reviewer_node**

In [ ]:
from pydantic import BaseModel, Field

# Structured output for reviewer_node
class ReviewResponse(BaseModel):
    decision: str = Field(description="Either 'PASS' or 'FAIL'")
    feedback: str = Field(description="Specific advice for the writer if FAIL")
    
reviewer_llm = llm.with_structured_output(ReviewResponse)

def reviewer_node(state: ResearchState):
    """Review final report and provide feedback"""
    current_count = state.get("loop_count", 0)
    print("--- Reviewing final report ---")
    question = state['question']
    report = state['final_report']
    prompt = f"""You are a strict Senior Research Quality Auditor. 
	Your job is to criticize the following research report for accuracy, completeness, and structure.

	Original Question: {question}
	Report to Review: {report}

	Critically evaluate:
	1. Does this fully answer the user's question?
	2. Is the formatting professional?
	3. Are there any logical gaps?

	Output your response in this EXACT JSON format:
	{{
		"decision": "PASS" or "FAIL",
		"feedback": "If FAIL, list exactly what needs to be improved. If PASS, leave empty."
	}}
	"""
    response = reviewer_llm.invoke(prompt)
    return {"feedback": f"{response.decision}: {response.feedback}", "loop_count": current_count + 1}

**Agentic Builder graph/flow**

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

def should_continue(state: ResearchState):
    """Router: Decides whether to finish or rewrite."""
    if state.get("loop_count", 0) >= 3:
        print("--- MAX ITERATIONS REACHED: FORCING END ---")
        return END
    if "PASS" in state["feedback"]:
        return END
    else:
        print(f"--- FEEDBACK: {state['feedback']} ---")
        return "writer" # Loop back to writer

# Building the Graph
builder = StateGraph(ResearchState)

builder.add_node("planner", planner_node)
builder.add_node("search", search_node)
builder.add_node("writer", writer_node)
builder.add_node("reviewer", reviewer_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "search")
builder.add_edge("search", "writer")
builder.add_edge("writer", "reviewer")

# This is where the loop happens!
builder.add_conditional_edges(
    "reviewer", 
    should_continue, 
    {
        END: END,           # If should_continue returns END, go to END
        "writer": "writer"  # If should_continue returns "writer", go to writer
    }
)

app = builder.compile(checkpointer=memory)

**Test run**

In [ ]:
inputs = {"question": "What are the latest breakthroughs in solid-state batteries as of late 2025?"}
config = {"configurable": {"thread_id": "research_session_1"}}
user_input = {}
for output in app.stream(inputs, config=config):
    for key, value in output.items():
        print(f"\nNode '{key}' finished.")
        if "plan" in value:
            print("Current Plan:")
            for i, step in enumerate(value['plan'], 1):
                print(f"{i}. {step}")
        if key == "search":
            print(f"Successfully retrieved data for {len(value['report_sections'])} topics.")
        if key == 'reviewer':
            print(f"Review result: {value['feedback']}")
        if key == "writer":
            print(f"Successfully created professional markdown report.\n{(value['final_report'])}")

In [17]:
from langgraph.types import Command

# 1. Setup
inputs = {"question": "What are the latest breakthroughs in solid-state batteries as of late 2025?"}
config = {"configurable": {"thread_id": "research_session_1"}}

# 2. Run until Interrupt
# This will return a dictionary once it hits the 'interrupt' line
print("--- Starting Research ---")
result = app.invoke(inputs, config=config)

# 3. Check if we are paused
snapshot = app.get_state(config)
if snapshot.next:
    # Access the plan proposed by the AI
    # This matches the key 'plan' you put in your interrupt() call
    proposed_plan = snapshot.tasks[0].interrupts[0].value['plan']
    
    print("\n[AI PROPOSED PLAN]")
    for i, step in enumerate(proposed_plan, 1):
        print(f"{i}. {step}")
    print("\n--- Graph is now PAUSED. Move to the next cell to resume. ---")

--- Starting Research ---

[AI PROPOSED PLAN]
1. Identify the primary technical hurdles and performance limitations currently faced by solid-state battery technology (e.g., electrolyte conductivity, interface stability, manufacturing cost, dendrite formation).
2. Search for academic publications, patent applications, and industry reports from 2023-2025 detailing advancements in solid-state battery components, such as novel electrolyte materials (e.g., sulfide, oxide, polymer), electrode compositions, or cell architectures.
3. Investigate major industry players and research institutions (e.g., Toyota, QuantumScape, Samsung, Solid Power) for announced milestones, technological demonstrations, and strategic roadmaps projecting significant progress or commercialization efforts by late 2025.
4. Analyze the potential impact of these advancements on key battery performance metrics, including energy density, power density, cycle life, safety features, and manufacturing scalability, as projecte

In [18]:
from langgraph.types import Command

# Provide feedback: {} for OK, or {"updated_plan": [...]} to edit
user_feedback = {} 

print("--- Resuming Graph ---")

# Use a loop to ensure the graph runs through all remaining nodes (search, writer, reviewer)
# Command(resume) wakes it up; the loop continues until END
for update in app.stream(Command(resume=user_feedback), config=config):
    node_name = list(update.keys())[0]
    print(f"Node '{node_name}' finished.")

# 4. Fetch the FINAL state values from the checkpointer
final_state = app.get_state(config).values

print("\n" + "="*50)
print("FINAL RESEARCH REPORT")
print("="*50 + "\n")
print(final_state.get("final_report", "Report not found. Check if the graph reached the end."))

--- Resuming Graph ---
Node '__interrupt__' finished.

FINAL RESEARCH REPORT

Report not found. Check if the graph reached the end.


**Flow chart**

In [ ]:
from IPython.display import Image, display

# Ensure graph is compiled (e.g., app = builder.compile())
try:
    # We use draw_mermaid_png to get a visual representation
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    # If the PNG generation fails (usually due to internet/API issues), 
    # use ASCII as a backup!
    print("Could not generate PNG. Falling back to ASCII:")
    print(app.get_graph().draw_ascii())